# SWE and SME from scratch — symbolic walkthrough

Derivation of the Shallow-Water Equations (SWE, level=0) and the Shallow
Moment Equations (SME, level>0) directly from incompressible Navier-Stokes,
using only base primitives:

* ``Expression.apply(...)`` on an ``_EquationProxy`` (``model.x_momentum.apply(...)``)
  — returns the proxy so calls chain.
* ``Integrate(var, lower, upper, method="auto"|"analytical"|"leibniz"|"fundamental_theorem"|"direct")``
  — unified integration Operation.  Per-term dispatch through
  ``Expression.depth_integrate`` for auto/leibniz/fundamental; whole-expression
  ``sympy.integrate`` for analytical.  Boundary terms are kept as
  ``Subs(f, var, bound)`` and resolved later via plain substitution.
* ``model.z_momentum.solve_for(state.p)`` — returns an Expression whose
  ``_as_relation`` is consumed by ``apply`` as a substitution.
* Substitution dicts for the remaining BC closures
  (atmospheric pressure at the free surface, zero surface/bottom
  tangential stress, kinematic BCs on w).
* ``Newtonian`` Relation kept as a convenience for the stress tensor.

No ``DepthIntegrate``, ``HydrostaticPressure``, ``ApplyKinematicBCs``,
``StressFreeSurface``, ``ZeroAtmosphericPressure``, ``SimplifyIntegrals``
shortcuts.  Every step is explicit.

## Imports

In [1]:
import sympy as sp

from zoomy_core.model.models.ins_generator import (
    StateSpace, FullINS, Integrate, Newtonian,
)
from zoomy_core.model.models.sme_model import hydrostatic_scaling

## Step 1 — Start from the raw Navier-Stokes system

In [2]:
state = StateSpace(dimension=2)          # (t, x, z)
model = FullINS(state)
model.describe()

**INS** (continuity, x_momentum, z_momentum)

**continuity:**
$$
\frac{\partial}{\partial x} u{\left(t,x,z \right)} + \frac{\partial}{\partial z} w{\left(t,x,z \right)} = 0
$$

**x_momentum:**
$$
\begin{aligned}
  & \underbrace{\frac{\partial}{\partial t} u{\left(t,x,z \right)}}_{temporal} \\
  & + \underbrace{\frac{\partial}{\partial z} u{\left(t,x,z \right)} w{\left(t,x,z \right)} + \frac{\partial}{\partial x} u^{2}{\left(t,x,z \right)}}_{convection} \\
  & + \underbrace{\frac{\frac{\partial}{\partial x} p{\left(t,x,z \right)}}{\rho}}_{pressure} \\
  & \underbrace{- \frac{\frac{\partial}{\partial x} \tau_{xx}{\left(t,x,z \right)} + \frac{\partial}{\partial z} \tau_{xz}{\left(t,x,z \right)}}{\rho}}_{stress}
  &= 0
\end{aligned}
$$

**z_momentum:**
$$
\begin{aligned}
  & \underbrace{\frac{\partial}{\partial t} w{\left(t,x,z \right)}}_{temporal} \\
  & + \underbrace{\frac{\partial}{\partial x} u{\left(t,x,z \right)} w{\left(t,x,z \right)} + \frac{\partial}{\partial z} w^{2}{\left(t,x,z \right)}}_{convection} \\
  & + \underbrace{\frac{\frac{\partial}{\partial z} p{\left(t,x,z \right)}}{\rho}}_{pressure} \\
  & \underbrace{- \frac{\frac{\partial}{\partial x} \tau_{zx}{\left(t,x,z \right)} + \frac{\partial}{\partial z} \tau_{zz}{\left(t,x,z \right)}}{\rho}}_{stress} \\
  & + \underbrace{g}_{source}
  &= 0
\end{aligned}
$$


## Step 2 — Hydrostatic assumption on z-momentum

Set $w = 0$, $\tau_{zz} = \tau_{xz} = \tau_{zx} = 0$ inside z-momentum.
`.apply()` returns the proxy so `.simplify()` chains.

In [3]:
model.z_momentum.apply(hydrostatic_scaling(state)).simplify()
model.z_momentum.describe()

**z_momentum** (2 terms)

$$
\begin{aligned}
  & \underbrace{\frac{\frac{\partial}{\partial z} p{\left(t,x,z \right)}}{\rho}}_{pressure} \\
  & + \underbrace{g}_{source}
  &= 0
\end{aligned}
$$

## Step 3 — Integrate z-momentum analytically to get $p(z)$

Integrate $g + \partial_z p / \rho = 0$ from the current depth $z$ up to the
free surface $\eta = b + h$ via analytical mode (whole-expression
``sympy.integrate``).

In [4]:
model.z_momentum.apply(
    Integrate(state.z, state.z, state.eta, method="analytical")
)
model.z_momentum.describe()

**z_momentum** (5 terms)

$$
\begin{aligned}
  & \underbrace{- \frac{p{\left(t,x,z \right)}}{\rho} + \frac{p{\left(t,x,b{\left(t,x \right)} + h{\left(t,x \right)} \right)}}{\rho}}_{pressure} \\
  & \underbrace{- g z + g \left(b{\left(t,x \right)} + h{\left(t,x \right)}\right)}_{source}
  &= 0
\end{aligned}
$$

## Step 4 — Atmospheric-pressure BC at the free surface

Close $p(\eta) = 0$ (atmospheric gauge) by a plain substitution dict.

In [5]:
model.z_momentum.apply({state.p.subs(state.z, state.eta): 0}).simplify()
model.z_momentum.describe()

**z_momentum** (4 terms)

$$
\begin{aligned}
  & \underbrace{- \frac{p{\left(t,x,z \right)}}{\rho}}_{pressure} \\
  & \underbrace{- g z + g b{\left(t,x \right)} + g h{\left(t,x \right)}}_{source}
  &= 0
\end{aligned}
$$

## Step 5 — Substitute the solved $p$ into x-momentum, then drop z-momentum

`model.z_momentum.solve_for(state.p)` returns an Expression that
``apply()`` consumes directly as ``{p: solution}``.

In [6]:
model.x_momentum.apply(model.z_momentum.solve_for(state.p)).simplify()
model.z_momentum.remove()
model.describe()

**INS** (continuity, x_momentum)

**continuity:**
$$
\frac{\partial}{\partial x} u{\left(t,x,z \right)} + \frac{\partial}{\partial z} w{\left(t,x,z \right)} = 0
$$

**x_momentum:**
$$
\begin{aligned}
  & \underbrace{\frac{\partial}{\partial t} u{\left(t,x,z \right)}}_{temporal} \\
  & + \underbrace{2 u{\left(t,x,z \right)} \frac{\partial}{\partial x} u{\left(t,x,z \right)} + u{\left(t,x,z \right)} \frac{\partial}{\partial z} w{\left(t,x,z \right)} + w{\left(t,x,z \right)} \frac{\partial}{\partial z} u{\left(t,x,z \right)}}_{convection} \\
  & + \underbrace{g \frac{\partial}{\partial x} b{\left(t,x \right)} + g \frac{\partial}{\partial x} h{\left(t,x \right)}}_{pressure} \\
  & \underbrace{- \frac{\frac{\partial}{\partial x} \tau_{xx}{\left(t,x,z \right)}}{\rho} - \frac{\frac{\partial}{\partial z} \tau_{xz}{\left(t,x,z \right)}}{\rho}}_{stress}
  &= 0
\end{aligned}
$$


## Step 6 — Newtonian constitutive model

Kept as a convenience Relation.  Substitutes
$\tau_{ij} = \mu(\partial_j u_i + \partial_i u_j)$ in every equation.

In [7]:
newton = Newtonian(state)
for name in list(model.equations.keys()):
    model.equations[name] = model.equations[name].apply(newton).simplify()
model.describe()

**INS** (continuity, x_momentum)

**continuity:**
$$
\frac{\partial}{\partial x} u{\left(t,x,z \right)} + \frac{\partial}{\partial z} w{\left(t,x,z \right)} = 0
$$

**x_momentum:**
$$
\begin{aligned}
  & \underbrace{\frac{\partial}{\partial t} u{\left(t,x,z \right)}}_{temporal} \\
  & + \underbrace{2 u{\left(t,x,z \right)} \frac{\partial}{\partial x} u{\left(t,x,z \right)} + u{\left(t,x,z \right)} \frac{\partial}{\partial z} w{\left(t,x,z \right)} + w{\left(t,x,z \right)} \frac{\partial}{\partial z} u{\left(t,x,z \right)}}_{convection} \\
  & + \underbrace{g \frac{\partial}{\partial x} b{\left(t,x \right)} + g \frac{\partial}{\partial x} h{\left(t,x \right)}}_{pressure} \\
  & \underbrace{- 2 \nu \frac{\partial^{2}}{\partial x^{2}} u{\left(t,x,z \right)} - \nu \frac{\partial^{2}}{\partial z^{2}} u{\left(t,x,z \right)} - \nu \frac{\partial^{2}}{\partial z\partial x} w{\left(t,x,z \right)}}_{stress}
  &= 0
\end{aligned}
$$


## Step 7 — Depth-integrate continuity and x-momentum from $b$ to $b+h$

One ``Integrate`` call with ``method="auto"`` per equation — per-term dispatch
picks Leibniz for $\partial_x$ and the fundamental theorem for $\partial_z$.

In [8]:
for name in list(model.equations.keys()):
    model.equations[name] = model.equations[name].apply(
        Integrate(state.z, state.b, state.eta, method="auto")
    )
model.continuity.describe()

**continuity** (5 terms)

$$
- \frac{\partial}{\partial x} \left(b{\left(t,x \right)} + h{\left(t,x \right)}\right) \left. u{\left(t,x,z \right)} \right|_{\substack{ z=b{\left(t,x \right)} + h{\left(t,x \right)} }} + \frac{\partial}{\partial x} b{\left(t,x \right)} \left. u{\left(t,x,z \right)} \right|_{\substack{ z=b{\left(t,x \right)} }} + \frac{\partial}{\partial x} \int\limits_{b{\left(t,x \right)}}^{b{\left(t,x \right)} + h{\left(t,x \right)}} u{\left(t,x,z \right)}\, dz + \left. w{\left(t,x,z \right)} \right|_{\substack{ z=b{\left(t,x \right)} + h{\left(t,x \right)} }} - \left. w{\left(t,x,z \right)} \right|_{\substack{ z=b{\left(t,x \right)} }} = 0
$$

In [9]:
model.x_momentum.describe()

**x_momentum** (17 terms)

$$
\begin{aligned}
  & \underbrace{- \frac{\partial}{\partial t} \left(b{\left(t,x \right)} + h{\left(t,x \right)}\right) \left. u{\left(t,x,z \right)} \right|_{\substack{ z=b{\left(t,x \right)} + h{\left(t,x \right)} }} + \frac{\partial}{\partial t} b{\left(t,x \right)} \left. u{\left(t,x,z \right)} \right|_{\substack{ z=b{\left(t,x \right)} }} + \frac{\partial}{\partial t} \int\limits_{b{\left(t,x \right)}}^{b{\left(t,x \right)} + h{\left(t,x \right)}} u{\left(t,x,z \right)}\, dz}_{temporal} \\
  & \underbrace{- \frac{\partial}{\partial x} \left(b{\left(t,x \right)} + h{\left(t,x \right)}\right) \left. 2 u^{2}{\left(t,x,z \right)} \right|_{\substack{ z=b{\left(t,x \right)} + h{\left(t,x \right)} }} + \frac{\partial}{\partial x} b{\left(t,x \right)} \left. 2 u^{2}{\left(t,x,z \right)} \right|_{\substack{ z=b{\left(t,x \right)} }} + \frac{\partial}{\partial x} \int\limits_{b{\left(t,x \right)}}^{b{\left(t,x \right)} + h{\left(t,x \right)}} 2 u^{2}{\left(t,x,z \right)}\, dz + 2 \left. u{\left(t,x,z \right)} w{\left(t,x,z \right)} \right|_{\substack{ z=b{\left(t,x \right)} + h{\left(t,x \right)} }} - 2 \left. u{\left(t,x,z \right)} w{\left(t,x,z \right)} \right|_{\substack{ z=b{\left(t,x \right)} }}}_{convection} \\
  & \underbrace{- \frac{\partial}{\partial x} \left(b{\left(t,x \right)} + h{\left(t,x \right)}\right) \left. g b{\left(t,x \right)} \right|_{\substack{ z=b{\left(t,x \right)} + h{\left(t,x \right)} }} - \frac{\partial}{\partial x} \left(b{\left(t,x \right)} + h{\left(t,x \right)}\right) \left. g h{\left(t,x \right)} \right|_{\substack{ z=b{\left(t,x \right)} + h{\left(t,x \right)} }} + \frac{\partial}{\partial x} b{\left(t,x \right)} \left. g b{\left(t,x \right)} \right|_{\substack{ z=b{\left(t,x \right)} }} + \frac{\partial}{\partial x} b{\left(t,x \right)} \left. g h{\left(t,x \right)} \right|_{\substack{ z=b{\left(t,x \right)} }} + \frac{\partial}{\partial x} \int\limits_{b{\left(t,x \right)}}^{b{\left(t,x \right)} + h{\left(t,x \right)}} g b{\left(t,x \right)}\, dz + \frac{\partial}{\partial x} \int\limits_{b{\left(t,x \right)}}^{b{\left(t,x \right)} + h{\left(t,x \right)}} g h{\left(t,x \right)}\, dz}_{pressure} \\
  & + \underbrace{\int\limits_{b{\left(t,x \right)}}^{b{\left(t,x \right)} + h{\left(t,x \right)}} \left(- 2 \nu \frac{\partial^{2}}{\partial x^{2}} u{\left(t,x,z \right)}\right)\, dz + \int\limits_{b{\left(t,x \right)}}^{b{\left(t,x \right)} + h{\left(t,x \right)}} \left(- \nu \frac{\partial^{2}}{\partial z^{2}} u{\left(t,x,z \right)}\right)\, dz + \int\limits_{b{\left(t,x \right)}}^{b{\left(t,x \right)} + h{\left(t,x \right)}} \left(- \nu \frac{\partial^{2}}{\partial z\partial x} w{\left(t,x,z \right)}\right)\, dz}_{stress}
  &= 0
\end{aligned}
$$

## Step 8 — Resolve $w$ boundary terms via the kinematic BCs

Two substitutions — the kinematic BCs at the bottom and at the surface —
written as plain dicts.  The `Subs(...)` shapes that Step 7 left in the
equation are the same expressions these dicts key on, so substitution is
direct.

In [10]:
w_at_b = state.w.subs(state.z, state.b)
w_at_eta = state.w.subs(state.z, state.eta)
u_at_b = state.u.subs(state.z, state.b)
u_at_eta = state.u.subs(state.z, state.eta)

kinematic_bcs = {
    w_at_b:   sp.Derivative(state.b, state.t)   + u_at_b   * sp.Derivative(state.b, state.x),
    w_at_eta: sp.Derivative(state.eta, state.t) + u_at_eta * sp.Derivative(state.eta, state.x),
}

for name in list(model.equations.keys()):
    model.equations[name] = model.equations[name].apply(kinematic_bcs).simplify()

model.x_momentum.describe()

**x_momentum** (17 terms)

$$
\begin{aligned}
  & \underbrace{- u{\left(t,x,b{\left(t,x \right)} + h{\left(t,x \right)} \right)} \frac{\partial}{\partial t} b{\left(t,x \right)} - u{\left(t,x,b{\left(t,x \right)} + h{\left(t,x \right)} \right)} \frac{\partial}{\partial t} h{\left(t,x \right)} + u{\left(t,x,b{\left(t,x \right)} \right)} \frac{\partial}{\partial t} b{\left(t,x \right)} + \frac{\partial}{\partial t} \int\limits_{b{\left(t,x \right)}}^{b{\left(t,x \right)} + h{\left(t,x \right)}} u{\left(t,x,z \right)}\, dz}_{temporal} \\
  & \underbrace{- 2 u^{2}{\left(t,x,b{\left(t,x \right)} + h{\left(t,x \right)} \right)} \frac{\partial}{\partial x} b{\left(t,x \right)} - 2 u^{2}{\left(t,x,b{\left(t,x \right)} + h{\left(t,x \right)} \right)} \frac{\partial}{\partial x} h{\left(t,x \right)} + 2 u{\left(t,x,b{\left(t,x \right)} + h{\left(t,x \right)} \right)} w{\left(t,x,b{\left(t,x \right)} + h{\left(t,x \right)} \right)} + 2 u^{2}{\left(t,x,b{\left(t,x \right)} \right)} \frac{\partial}{\partial x} b{\left(t,x \right)} - 2 u{\left(t,x,b{\left(t,x \right)} \right)} w{\left(t,x,b{\left(t,x \right)} \right)} + \frac{\partial}{\partial x} \int\limits_{b{\left(t,x \right)}}^{b{\left(t,x \right)} + h{\left(t,x \right)}} 2 u^{2}{\left(t,x,z \right)}\, dz}_{convection} \\
  & \underbrace{- g b{\left(t,x \right)} \frac{\partial}{\partial x} h{\left(t,x \right)} - g h{\left(t,x \right)} \frac{\partial}{\partial x} h{\left(t,x \right)} + \frac{\partial}{\partial x} \int\limits_{b{\left(t,x \right)}}^{b{\left(t,x \right)} + h{\left(t,x \right)}} g b{\left(t,x \right)}\, dz + \frac{\partial}{\partial x} \int\limits_{b{\left(t,x \right)}}^{b{\left(t,x \right)} + h{\left(t,x \right)}} g h{\left(t,x \right)}\, dz}_{pressure} \\
  & + \underbrace{\int\limits_{b{\left(t,x \right)}}^{b{\left(t,x \right)} + h{\left(t,x \right)}} \left(- 2 \nu \frac{\partial^{2}}{\partial x^{2}} u{\left(t,x,z \right)}\right)\, dz + \int\limits_{b{\left(t,x \right)}}^{b{\left(t,x \right)} + h{\left(t,x \right)}} \left(- \nu \frac{\partial^{2}}{\partial z^{2}} u{\left(t,x,z \right)}\right)\, dz + \int\limits_{b{\left(t,x \right)}}^{b{\left(t,x \right)} + h{\left(t,x \right)}} \left(- \nu \frac{\partial^{2}}{\partial z\partial x} w{\left(t,x,z \right)}\right)\, dz}_{stress}
  &= 0
\end{aligned}
$$

## Step 9 — Zero tangential stress at surface and bottom

Two plain dicts: stress-free surface ($\tau_{xz}|_\eta = 0$) and
zero tangential normal stress at both boundaries ($\tau_{xx}|_b = \tau_{xx}|_\eta = 0$).
Bottom shear stress ($\tau_{xz}|_b$) stays symbolic — close it with a
Navier-slip or no-slip assumption as a further `.apply({...})` step.

In [11]:
no_surface_shear = {state.tau["xz"].subs(state.z, state.eta): 0}
no_tangential_normal_stress = {
    state.tau["xx"].subs(state.z, state.b): 0,
    state.tau["xx"].subs(state.z, state.eta): 0,
}

for name in list(model.equations.keys()):
    model.equations[name] = (
        model.equations[name]
        .apply(no_surface_shear)
        .apply(no_tangential_normal_stress)
        .simplify()
    )

model.x_momentum.describe()

**x_momentum** (17 terms)

$$
\begin{aligned}
  & \underbrace{- u{\left(t,x,b{\left(t,x \right)} + h{\left(t,x \right)} \right)} \frac{\partial}{\partial t} b{\left(t,x \right)} - u{\left(t,x,b{\left(t,x \right)} + h{\left(t,x \right)} \right)} \frac{\partial}{\partial t} h{\left(t,x \right)} + u{\left(t,x,b{\left(t,x \right)} \right)} \frac{\partial}{\partial t} b{\left(t,x \right)} + \frac{\partial}{\partial t} \int\limits_{b{\left(t,x \right)}}^{b{\left(t,x \right)} + h{\left(t,x \right)}} u{\left(t,x,z \right)}\, dz}_{temporal} \\
  & \underbrace{- 2 u^{2}{\left(t,x,b{\left(t,x \right)} + h{\left(t,x \right)} \right)} \frac{\partial}{\partial x} b{\left(t,x \right)} - 2 u^{2}{\left(t,x,b{\left(t,x \right)} + h{\left(t,x \right)} \right)} \frac{\partial}{\partial x} h{\left(t,x \right)} + 2 u{\left(t,x,b{\left(t,x \right)} + h{\left(t,x \right)} \right)} w{\left(t,x,b{\left(t,x \right)} + h{\left(t,x \right)} \right)} + 2 u^{2}{\left(t,x,b{\left(t,x \right)} \right)} \frac{\partial}{\partial x} b{\left(t,x \right)} - 2 u{\left(t,x,b{\left(t,x \right)} \right)} w{\left(t,x,b{\left(t,x \right)} \right)} + \frac{\partial}{\partial x} \int\limits_{b{\left(t,x \right)}}^{b{\left(t,x \right)} + h{\left(t,x \right)}} 2 u^{2}{\left(t,x,z \right)}\, dz}_{convection} \\
  & \underbrace{- g b{\left(t,x \right)} \frac{\partial}{\partial x} h{\left(t,x \right)} - g h{\left(t,x \right)} \frac{\partial}{\partial x} h{\left(t,x \right)} + \frac{\partial}{\partial x} \int\limits_{b{\left(t,x \right)}}^{b{\left(t,x \right)} + h{\left(t,x \right)}} g b{\left(t,x \right)}\, dz + \frac{\partial}{\partial x} \int\limits_{b{\left(t,x \right)}}^{b{\left(t,x \right)} + h{\left(t,x \right)}} g h{\left(t,x \right)}\, dz}_{pressure} \\
  & + \underbrace{\int\limits_{b{\left(t,x \right)}}^{b{\left(t,x \right)} + h{\left(t,x \right)}} \left(- 2 \nu \frac{\partial^{2}}{\partial x^{2}} u{\left(t,x,z \right)}\right)\, dz + \int\limits_{b{\left(t,x \right)}}^{b{\left(t,x \right)} + h{\left(t,x \right)}} \left(- \nu \frac{\partial^{2}}{\partial z^{2}} u{\left(t,x,z \right)}\right)\, dz + \int\limits_{b{\left(t,x \right)}}^{b{\left(t,x \right)} + h{\left(t,x \right)}} \left(- \nu \frac{\partial^{2}}{\partial z\partial x} w{\left(t,x,z \right)}\right)\, dz}_{stress}
  &= 0
\end{aligned}
$$

## Step 10 — Bottom stress closure (Navier slip)

$\tau_{xz}|_b = \rho\,(\lambda/\tau_c)\,u|_b$ — the only non-trivial
boundary term still present.  Plain substitution dict.

After this step the depth-integrated PDE has only ``u(t,x,z)``, its
boundary evaluations, and the remaining volume ``Integral(...)`` terms.
Projection onto a vertical basis (separate step — level=0 for SWE,
level>0 for SME) produces the final closed equations.

In [12]:
lamda = sp.Symbol("lamda", positive=True)
tau_c = sp.Symbol("tau_c", positive=True)
friction_closure = {
    state.tau["xz"].subs(state.z, state.b): state.rho * (lamda / tau_c) * u_at_b,
}

for name in list(model.equations.keys()):
    model.equations[name] = model.equations[name].apply(friction_closure).simplify()

model.x_momentum.describe()

**x_momentum** (17 terms)

$$
\begin{aligned}
  & \underbrace{- u{\left(t,x,b{\left(t,x \right)} + h{\left(t,x \right)} \right)} \frac{\partial}{\partial t} b{\left(t,x \right)} - u{\left(t,x,b{\left(t,x \right)} + h{\left(t,x \right)} \right)} \frac{\partial}{\partial t} h{\left(t,x \right)} + u{\left(t,x,b{\left(t,x \right)} \right)} \frac{\partial}{\partial t} b{\left(t,x \right)} + \frac{\partial}{\partial t} \int\limits_{b{\left(t,x \right)}}^{b{\left(t,x \right)} + h{\left(t,x \right)}} u{\left(t,x,z \right)}\, dz}_{temporal} \\
  & \underbrace{- 2 u^{2}{\left(t,x,b{\left(t,x \right)} + h{\left(t,x \right)} \right)} \frac{\partial}{\partial x} b{\left(t,x \right)} - 2 u^{2}{\left(t,x,b{\left(t,x \right)} + h{\left(t,x \right)} \right)} \frac{\partial}{\partial x} h{\left(t,x \right)} + 2 u{\left(t,x,b{\left(t,x \right)} + h{\left(t,x \right)} \right)} w{\left(t,x,b{\left(t,x \right)} + h{\left(t,x \right)} \right)} + 2 u^{2}{\left(t,x,b{\left(t,x \right)} \right)} \frac{\partial}{\partial x} b{\left(t,x \right)} - 2 u{\left(t,x,b{\left(t,x \right)} \right)} w{\left(t,x,b{\left(t,x \right)} \right)} + \frac{\partial}{\partial x} \int\limits_{b{\left(t,x \right)}}^{b{\left(t,x \right)} + h{\left(t,x \right)}} 2 u^{2}{\left(t,x,z \right)}\, dz}_{convection} \\
  & \underbrace{- g b{\left(t,x \right)} \frac{\partial}{\partial x} h{\left(t,x \right)} - g h{\left(t,x \right)} \frac{\partial}{\partial x} h{\left(t,x \right)} + \frac{\partial}{\partial x} \int\limits_{b{\left(t,x \right)}}^{b{\left(t,x \right)} + h{\left(t,x \right)}} g b{\left(t,x \right)}\, dz + \frac{\partial}{\partial x} \int\limits_{b{\left(t,x \right)}}^{b{\left(t,x \right)} + h{\left(t,x \right)}} g h{\left(t,x \right)}\, dz}_{pressure} \\
  & + \underbrace{\int\limits_{b{\left(t,x \right)}}^{b{\left(t,x \right)} + h{\left(t,x \right)}} \left(- 2 \nu \frac{\partial^{2}}{\partial x^{2}} u{\left(t,x,z \right)}\right)\, dz + \int\limits_{b{\left(t,x \right)}}^{b{\left(t,x \right)} + h{\left(t,x \right)}} \left(- \nu \frac{\partial^{2}}{\partial z^{2}} u{\left(t,x,z \right)}\right)\, dz + \int\limits_{b{\left(t,x \right)}}^{b{\left(t,x \right)} + h{\left(t,x \right)}} \left(- \nu \frac{\partial^{2}}{\partial z\partial x} w{\left(t,x,z \right)}\right)\, dz}_{stress}
  &= 0
\end{aligned}
$$

## What comes next

* **SWE (level=0)** — project with a constant vertical profile: every
  ``u(t,x,z) → u_mean(t,x)``, boundary evaluations become the same
  constant, ``∫ u dz = h · u_mean``.  A few `.apply({...})` substitutions
  complete the reduction.
* **SME (level≥1)** — project with ``u(t,x,z) = sum_k α_k(t,x) φ_k(ζ)``
  and Galerkin-test against each basis function.  Today this is
  ``Expression.project_onto_basis(basis, level, field_map, var, test_mode=l)``;
  it only rewrites ``Integral`` nodes, so a small companion substitution
  for surface/bottom ``u`` evaluations is needed (covered in the next
  file, once `project_onto_basis` is extended).

The notebook above is the derivation spine — clean, reviewable, built
entirely from `apply` / `Integrate` / substitution dicts / `solve_for` /
`Newtonian`.  No convenience Relation beyond `Newtonian` and
`hydrostatic_scaling` is used.